### Testing graph construction and comparing results for small language models and large language models

This notebook evaluates the following pipelines to determine the most effective approach for graph construction:
1. Basic graph extraction with small language models
2. Basic graph extraction with large language models
3. Graph extraction with pre-stage of extracting entities names with small language models
4. Graph extraction with pre-stage of extracting entities names with large language models

In [ ]:
#imports

import os
import sys
import tqdm
import pandas as pd
import numpy as np
import logging
import warnings
import json
from typing import Any, Dict, List, Optional, Literal
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_community.vectorstores import FAISS
from pandas import json_normalize
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

#some important stuff setup

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)
sys.path.insert(0, project_root)

results_dir = os.path.join(project_root, "assets", "outputs", "test_results")

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("faiss").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

#this nir imports

from nir.llm.manager import ModelManager
from nir.llm.providers import ModelConfig

from nir.tests.test_datasets import GRAPH_TEST_DATASET_SMALL, GRAPH_TEST_DATASET_SMALL_FROM_NODES, GRAPH_TEST_DATASET_LARGE, GRAPH_TEST_DATASET_LARGE_FROM_NODES
from nir.tests.evaluator import analyze_generation, compare_pipelines_directional, compute_effect_size
from nir.tests.metrics import compute_mauve_en, compute_mauve_ru, compute_self_bleu, evaluate_ragas_metrics_batch

from nir.graph.graph_storages.networkx_graph import NetworkXGraph
from nir.core.context_retriever import form_context_with_llm, form_context_without_llm
from nir.core.answers_generator import generate_plan
from nir.core.answers_generator import generate_answer_based_on_plan
from nir.core.answers_generator import generate_answer_based_on_context

In [ ]:
#load datasets

graphs_small = GRAPH_TEST_DATASET_SMALL
graphs_small_from_nodes = GRAPH_TEST_DATASET_SMALL_FROM_NODES
graphs_large = GRAPH_TEST_DATASET_LARGE
graphs_large_from_nodes = GRAPH_TEST_DATASET_LARGE_FROM_NODES

**Testing basic LLM**